# **Question 4**
Write the Viterbi algorithm to implment Nature Primer.

Here are some suggestions:

a. You can begin by first defining all the parameters, such as states, transition matrix, and emmision matrix etc.

b. You can write a function to exactly calculate the values mentioned in the primer, for example, you can define a function get_log_prob_of_a_given_path ("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA"). This should output -41.22.

By doing the above two, you earn 1 mark.

Now, you have to implement this in real to get max likely path that would emmit the observed sequence. You MUST note that maximum likely path will just be Es, but that is okay. Implementation is the key.

In [1]:
import numpy as np
import math

# Define HMM components
states = ['E', '5', 'I']
nucleotides = ['A', 'C', 'G', 'T']

initial_probabilities = {'E': 1.0, '5': 0.0, 'I': 0.0}

transition_probabilities = {
    'Start': {'E': 1.0, '5': 0.0, 'I': 0.0, 'End': 0.0},
    'E':     {'E': 0.9, '5': 0.1, 'I': 0.0, 'End': 0.0},
    '5':     {'E': 0.0, '5': 0.0, 'I': 1.0, 'End': 0.0},
    'I':     {'E': 0.0, '5': 0.0, 'I': 0.9, 'End': 0.1}
}

emission_probs = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'C': 0.00, 'G': 0.95, 'T': 0.00},
    'I': {'A': 0.40, 'C': 0.10, 'G': 0.10, 'T': 0.40}
}

def log(x):
    return -math.inf if x == 0 else math.log(x)

def get_log_prob_of_a_given_path(state_path, observed_sequence):
    if len(state_path) != len(observed_sequence):
        raise ValueError("State path and observed sequence must be the same length")

    log_prob = 0.0
    prev_state = 'Start'

    for state, obs in zip(state_path, observed_sequence):
        if obs not in nucleotides:
            raise ValueError(f"Invalid nucleotide '{obs}' in observed sequence")

        trans_prob = transition_probabilities[prev_state][state]
        emit_prob = emission_probs[state][obs]

        log_prob += log(trans_prob) + log(emit_prob)
        prev_state = state

    if prev_state == 'I':
        log_prob += log(transition_probabilities['I']['End'])

    return log_prob

# Example usage
state_path = "EEEEEEEEEEEEEEEEEE5IIIIIII"
observed_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"
log_probability = get_log_prob_of_a_given_path(state_path, observed_sequence)

print("Log probability of the given path:", log_probability)


Log probability of the given path: -41.21967768602254


In [2]:
import numpy as np
import math

def log(x):
    return -math.inf if x == 0 else math.log(x)

def viterbiAlgo(observed_sequence):
    num_states = len(states)
    num_observations = len(observed_sequence)

    viterbi_matrix = np.full((num_states, num_observations), -np.inf)
    backpointers = np.zeros((num_states, num_observations), dtype=int)

    state_to_index = {s: i for i, s in enumerate(states)}

    # Initialization
    for i, state in enumerate(states):
        viterbi_matrix[i, 0] = log(initial_probabilities[state]) + \
                               log(emission_probs[state][observed_sequence[0]])

    # Recursion
    for t in range(1, num_observations):
        for curr_index, curr_state in enumerate(states):
            max_log_prob = -math.inf
            best_prev_idx = 0
            for prev_index, prev_state in enumerate(states):
                trans_log_prob = log(transition_probabilities[prev_state][curr_state])
                total_log_prob = viterbi_matrix[prev_index, t-1] + trans_log_prob
                if total_log_prob > max_log_prob:
                    max_log_prob = total_log_prob
                    best_prev_idx = prev_index
            viterbi_matrix[curr_index, t] = max_log_prob + \
                                            log(emission_probs[curr_state][observed_sequence[t]])
            backpointers[curr_index, t] = best_prev_idx

    # Termination and backtracking
    best_last_state_index = np.argmax(viterbi_matrix[:, -1])
    best_path = [states[best_last_state_index]]

    for t in range(num_observations - 1, 0, -1):
        best_last_state_index = backpointers[best_last_state_index, t]
        best_path.insert(0, states[best_last_state_index])

    best_log_prob = np.max(viterbi_matrix[:, -1])
    return best_path, best_log_prob

# Example usage
observed_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"
best_path, log_prob = viterbiAlgo(observed_sequence)

print(f"Most probable path is: {''.join(best_path)}")
print(f"Log probability of the path: {log_prob}")


Most probable path is: EEEEEEEEEEEEEEEEEEEEEEEEEE
Log probability of the path: -38.677666280562796
